In [2]:
from pathlib import Path
import pandas as pd

url_base = "https://data.sports.gouv.fr/api/explore/v2.1/catalog/datasets/{}/exports/csv?delimiter=%3B"

df_histo = pd.read_csv(url_base.format("lics_histo"), sep=";")
df_2025_dep = pd.read_csv(url_base.format("lics_dep_age_sexe"), sep=";")
df_dep_ref = pd.read_csv(url_base.format("lics_dep"), sep=";")

for nom, df in [("df_histo", df_histo), ("df_2025_dep", df_2025_dep), ("df_dep_ref", df_dep_ref)]:
    print(nom, df.shape, list(df.columns))

df_histo (3204, 11) ['code_fede_2025', 'code_fede_ref', 'federation', 'annee', 'evol_20ans', 'evol_10ans', 'evol_5ans', 'lics', 'lics_100_2002', 'lics_100_2016', 'fede_gp']
df_2025_dep (459360, 16) ['code_fede_2025', 'code_fede_ref', 'sexe', 'trage_full', 'trage', 'dep', 'dep_l', 'newreg', 'newreg_l', 'lics', 'pop', 'lics_pop', 'rank_fede_dep', 'lics_pop_fr', 'is', 'fede_gp']
df_dep_ref (12760, 17) ['code_fede_2025', 'code_fede_ref', 'rank_fede_dep', 'newreg', 'newreg_l', 'dep', 'dep_l', 'lics', 'pop', 'lics_pop', 'lics_pop_fr', 'is', 'dep_name', 'reg_name', 'federations_sportives', 'federation_court', 'annee']


In [3]:
print(df_histo.columns)

Index(['code_fede_2025', 'code_fede_ref', 'federation', 'annee', 'evol_20ans',
       'evol_10ans', 'evol_5ans', 'lics', 'lics_100_2002', 'lics_100_2016',
       'fede_gp'],
      dtype='str')


In [4]:
for col in df_histo.columns:
    df_histo[col] = df_histo[col].astype("string").str.strip()

In [5]:
for val in sorted(df_histo["federation"].dropna().unique()):
    print(val)

0. Toutes fédés (champ constant 2016-2025)
1. Fédérations olympiques (champ constant 2016-2025)
2. Fédérations délégataires non-olympiques (champ constant 2016-2025)
4. Fédérations affinitaires (champ constant 2016-2025)
5. Fédérations para-sportives (champ constant 2016-2025)
6. Fédérations scolaires (champ constant 2016-2025)
Association française de snowboard
Centre nautique des Glénans
Fédération Française Aéronautique
Fédération Française Handisport
Fédération Française Maccabi
Fédération Française Motonautique
Fédération Française Omnisports des Personnels de l'Éducation Nationale (2FOPEN)
Fédération Française Sportive de Twirling Bâton
Fédération Française Sports Pour Tous
Fédération Française d'Athlétisme
Fédération Française d'Aviron
Fédération Française d'Aéromodélisme
Fédération Française d'Aérostation
Fédération Française d'Aïkido Aïkibudo Kinomichi et Disciplines Associées
Fédération Française d'Aïkido et de Budo
Fédération Française d'Escrime
Fédération Française d'Haltér

In [6]:
for val in sorted(df_histo["fede_gp"].dropna().unique()):
    print(val)

0. Toutes fédés
1. Olympique
2. Délégataire non-olympique
4. Affinitaire
5. Para-sport
6. Scolaire


In [7]:
for val in sorted(df_histo["code_fede_ref"].dropna().unique()):
    print(val)

101
102
103
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
135
136
137
138
140
141
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
223
224
226
227
228
229
230
231
233
234
235
236
237
238
239
240
241
242
243
244
245
248
249
250
251
254
255
256
257
258
260
261
263
264
265
266
267
268
269
270
271
402
403
404
405
406
407
408
409
410
411
412
413
414
415
417
418
420
421
422
423
424
425
426
501
502
503
601
602
603
604
605
TOT


In [8]:
df_histo.isna().sum()

code_fede_2025      0
code_fede_ref       0
federation          0
annee               0
evol_20ans        756
evol_10ans        636
evol_5ans         528
lics              402
lics_100_2002     685
lics_100_2016     573
fede_gp             0
dtype: int64

In [9]:
for col in ["code_fede_2025", "code_fede_ref", "federation"] :
    print(df_histo[df_histo["lics"].isna()][col].unique())

<ArrowStringArray>
['130', '131', '223', '230', '235', '236', '238', '239', '240', '260', '261',
 '263', '264', '265', '266', '267', '268', '269', '270', '412', '414', '418',
 '421', '422', '423', '424', '425', '426', '502', '227', '420']
Length: 31, dtype: string
<ArrowStringArray>
['130', '131', '223', '230', '235', '236', '238', '239', '240', '260', '261',
 '263', '264', '265', '266', '267', '268', '269', '270', '412', '414', '418',
 '421', '422', '423', '424', '425', '426', '502', '227', '420']
Length: 31, dtype: string
<ArrowStringArray>
[                                              'Association française de snowboard',
                                         'Fédération Française de Hockey sur Glace',
                        'Fédération fighting full contact et disciplines associées',
                                              'Fédération Française de kick boxing',
                        'Fédération Française de muaythaï et disciplines associées',
                          

In [10]:
# Regarde pour chaque fédération ayant des NaN : combien d'années manquent, et quelles années sont présentes
diag_na = (
    df_histo[df_histo["lics"].isna()]
    .groupby(["code_fede_2025", "federation"])["annee"]
    .agg(nb_na="count", premiere_annee_na="min", derniere_annee_na="max")
)
print(diag_na.to_string())

                                                                                                 nb_na premiere_annee_na derniere_annee_na
code_fede_2025 federation                                                                                                                 
130            Association française de snowboard                                                   20        2006-01-01        2025-01-01
131            Fédération Française de Hockey sur Glace                                              4        2002-01-01        2005-01-01
223            Fédération fighting full contact et disciplines associées                            12        2014-01-01        2025-01-01
227            Fédération Française de Jeu de Paume                                                  1        2025-01-01        2025-01-01
230            Fédération Française de kick boxing                                                  17        2009-01-01        2025-01-01
235            Fédération F

In [ ]:
#les valeurs manquantes concernent des fédérations qui n'existaient pas au début de la période étudiée
#et qui ont été créées après, ou bien des fédérations qui ont fusionné, qui se sont séparées, etc. On
#fait donc le choix de les laisser telles quelles, en "na"
df_histo_clean = pd.DataFrame({
    "code_fed": df_histo["code_fede_2025"].astype(str).str.strip().str.zfill(3),
    "federation": df_histo["federation"].astype(str).str.strip(),
    "annee": df_histo["annee"].astype(str).str[:4].astype(int),
    "licences": pd.to_numeric(df_histo["lics"], errors="coerce"),
    "categorie_fede": df_histo["fede_gp"].astype(str).str.replace(r"^\d+\.\s*", "", regex=True).str.strip(),
})

masque_totaux = (
    df_histo_clean["code_fed"].str.startswith("0")
    | df_histo_clean["code_fed"].str.upper().isin(["TOT", "TOTAL", "NAN"])
    | df_histo_clean["federation"].str.contains("champ constant|toutes fédés", case=False, na=False)
)
df_histo_clean = (
    df_histo_clean[~masque_totaux]
    .sort_values(["code_fed", "annee"])
    .reset_index(drop=True)
)

# vérification finale
print(df_histo_clean.info())
display(df_histo_clean.head())

ref_federations = (
    df_histo_clean[["code_fed", "federation", "categorie_fede"]]
    .drop_duplicates(subset=["code_fed"])
    .sort_values("code_fed")
    .reset_index(drop=True)
)


Path("data/processed").mkdir(parents=True, exist_ok=True)
df_histo_clean.to_csv("../data/processed/licences_histo_2002_2025.csv", sep=";", index=False)


<class 'pandas.DataFrame'>
RangeIndex: 3144 entries, 0 to 3143
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   code_fed        3144 non-null   str    
 1   federation      3144 non-null   str    
 2   annee           3144 non-null   int64  
 3   licences        2742 non-null   Float64
 4   categorie_fede  3144 non-null   str    
dtypes: Float64(1), int64(1), str(3)
memory usage: 327.7 KB
None


,code_fed,federation,annee,licences,categorie_fede
0,101,Fédération Française d'Athlétisme,2002,165857.0,Olympique
1,101,Fédération Française d'Athlétisme,2003,177509.0,Olympique
2,101,Fédération Française d'Athlétisme,2004,162946.0,Olympique
3,101,Fédération Française d'Athlétisme,2005,167007.0,Olympique
4,101,Fédération Française d'Athlétisme,2006,172198.0,Olympique
